In [30]:
import json
import os

In [ ]:
data_dir = '../evaluation/predictions/baseline'
wild_dir = "translationese/wild/enms"
data_files = os.listdir(data_dir)

In [ ]:
for data_file in data_files:
    if "cleaned" not in data_file:
        continue
    data = []
    with open(f"{data_dir}/{data_file}", 'r') as file:
        for line in file.readlines():
            data.append(json.loads(line))

    with open(f"{wild_dir}/{data_file.split('.')[0]}.jsonl", "w") as file:
        for item in data:
            line = {
                "source": item['src'],
                "foreignization": item['mt'],
                "domestication": item['ref'],
            }
            json.dump(line, file)
            file.write('\n')

In [3]:
os.makedirs(wild_dir, exist_ok=True)
for data_file in data_files:
    if "cleaned" not in data_file:
        continue
    os.rename(f"{data_dir}/{data_file}", f"{wild_dir}/{data_file.replace('_cleaned', '')}")

In [4]:
pointwise_dir = "translationese/wild/pointwise/enms"
pairwise_dir = "translationese/wild/pairwise/enms"

os.makedirs(pairwise_dir, exist_ok=True)

In [6]:
data = []
for data_file in os.listdir(pointwise_dir):
    for pair_file in os.listdir(pointwise_dir):
        if data_file == pair_file:
            continue
        files = [data_file, pair_file]
        dat = {}
        for i in range(len(files)):
            dat[i] = []
            with open(f"{pointwise_dir}/{files[i]}", 'r') as file:
                for line in file.readlines():
                    dat[i].append(json.loads(line))
        for d1, d2 in zip(dat[0], dat[1]):
            assert d1['source'] == d2['source']
            data.append({
                "source": d1['source'],
                "translation_A": d1['foreignization'],
                "translation_B": d2['foreignization']
            })

In [7]:
with open(f"{pairwise_dir}/pairwise.jsonl", "w") as file:
    for item in data:
        json.dump(item, file)
        file.write('\n')

In [3]:
def read_jsonl(split, path):
    with open(f"{path}/{split}.jsonl", 'r') as file:
        data = []
        for line in file.readlines():
            data.append(json.loads(line))
    return data

In [4]:
def write_jsonl(data, split, path):
    os.makedirs(path, exist_ok=True)
    with open(f"{path}/{split}.jsonl", 'w') as file:
        for item in data:
            json.dump(item, file)
            file.write('\n')

In [38]:
synthetic_dir = 'translationese/synthetic/enms'
datasets = os.listdir(synthetic_dir)

In [7]:
(676 / 2) // 6

56.0

In [11]:
with open("parallel_asian_treebank/translationese/enms/test.jsonl", "r") as file:
    test = []
    for line in file.readlines():
        test.append(json.loads(line))

with open("parallel_asian_treebank/translationese/enms/dev.jsonl", "r") as file:
    dev = []
    for line in file.readlines():
        dev.append(json.loads(line))

In [12]:
len(dev), len(test)

(320, 357)

In [19]:
diff = (len(test) - len(dev))
div = diff // 2

In [25]:
len(test[:-div-1]), len(test[-div:])

(338, 18)

In [23]:
len(dev) + len(test[-div:])

338

In [8]:
from functools import partial

In [29]:
for lang in datasets:
    data_path = synthetic_dir + '/' + lang
    open_jsonl = partial(read_jsonl, path=data_path)
    write_file = partial(write_jsonl, path=data_path)

    dev = open_jsonl("dev")
    test = open_jsonl("test")

    if len(test) > len(dev):
        diff = len(test) - len(dev)
        split = diff // 2
        rem = test[-div:]
        new_test = test[:-div-1]
        new_dev = dev + rem
        write_file(new_dev, 'dev')
        write_file(new_test, 'test')

In [33]:
train_size = 5382
valid_size = 338
test_size = 338

In [39]:
batches = {"train": [slice(int((5384 // len(datasets))) * i, int((5384 // len(datasets))) * (i+1)) for i in range(len(datasets))], "dev": [slice(int((338 // len(datasets))) * i, int((338 // len(datasets))) * (i+1)) for i in range(len(datasets))], "test": [slice(int((338 // len(datasets))) * i, int((338 // len(datasets))) * (i+1)) for i in range(len(datasets))]}

In [40]:
batches

{'train': [slice(0, 1076, None),
  slice(1076, 2152, None),
  slice(2152, 3228, None),
  slice(3228, 4304, None),
  slice(4304, 5380, None)],
 'dev': [slice(0, 67, None),
  slice(67, 134, None),
  slice(134, 201, None),
  slice(201, 268, None),
  slice(268, 335, None)],
 'test': [slice(0, 67, None),
  slice(67, 134, None),
  slice(134, 201, None),
  slice(201, 268, None),
  slice(268, 335, None)]}

In [41]:
dataset = {"train": [], "dev": [], "test": []}
for i, ds in enumerate(datasets):
    path = f"{synthetic_dir}/{ds}"
    open_jsonl = partial(read_jsonl, path=path)
    write_file = partial(write_jsonl, path=path)

    for split in dataset:
        data = open_jsonl(split)
        dataset[split] += data[batches[split][i]]
        

In [42]:
for split, data in dataset.items():
    write_jsonl(data, split, f"{synthetic_dir}/parallel_asian_treebank")

In [43]:
for split in dataset:
    print(len(dataset[split]))

5380
335
335
